# The Transformer

**Prerequisites**

- L08: Embeddings (`nn.Embedding`, learned dense representations)
- L09: RNNs (sequence modeling, hidden states, vanishing gradients, LSTM, GRU)
- Tokenization (previous notebook): converting text to integer sequences

**Outcomes**

- Understand the self-attention mechanism and derive its equations
- Understand multi-head attention and why it helps
- Understand positional encoding
- Build a complete decoder-only transformer (GPT-style) in PyTorch
- Understand pre-training objectives: next-token prediction vs. masked language modeling
- Train a small language model on real text and generate from it

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
np.set_printoptions(linewidth=140, precision=4, suppress=True)

%matplotlib inline

## From RNNs to Attention

In L09 we studied recurrent neural networks. An RNN processes a sequence one step at a time, updating a hidden state $h_t$ at each position:

$$h_t = f(x_t, h_{t-1})$$

This design creates two fundamental problems:

1. **Sequential bottleneck**: because $h_t$ depends on $h_{t-1}$, we cannot parallelize computation across timesteps during training. Each step must wait for the previous one to finish.

2. **Long-range dependencies**: information from early tokens must survive through a chain of hidden state updates to reach later tokens. Even with LSTMs and GRUs, information gets diluted over long sequences.

The **transformer** (Vaswani et al., 2017) solves both problems with a mechanism called **self-attention**. Instead of processing the sequence step-by-step, self-attention allows every token to directly attend to every other token in the sequence — all at once, in parallel.

Think of attention as a resource allocation mechanism: each token surveys the entire sequence and allocates its representational budget to the most relevant information, much like an agent in an economy allocating scarce resources to their highest-valued uses.

## Self-Attention

### Query, Key, Value

Given a sequence of $T$ token embeddings stacked into a matrix $X \in \mathbb{R}^{T \times d}$, self-attention computes three linear projections:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

where $W_Q, W_K \in \mathbb{R}^{d \times d_k}$ and $W_V \in \mathbb{R}^{d \times d_v}$ are learned weight matrices.

The intuition behind these three projections:

- **Query** $q_i$: "What information is token $i$ looking for?"
- **Key** $k_j$: "What information does token $j$ advertise?"
- **Value** $v_j$: "What information does token $j$ actually provide?"

The attention output for each token is a weighted sum of the value vectors, where the weights come from the compatibility between queries and keys. The formula is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Let's unpack this step by step.

### Step-by-step breakdown

Consider a concrete example with $T = 4$ tokens and $d_k = 3$.

**Step 1: Compute similarity scores.**

$$\text{scores} = QK^\top \in \mathbb{R}^{T \times T}$$

Each entry $\text{scores}_{ij} = q_i \cdot k_j$ measures how much token $i$'s query matches token $j$'s key — i.e., how relevant token $j$ is to token $i$.

**Step 2: Scale.**

$$\text{scaled\_scores} = \frac{\text{scores}}{\sqrt{d_k}}$$

We will explain the reason for this scaling shortly.

**Step 3: Normalize with softmax.**

$$\text{weights}_{ij} = \frac{\exp(\text{scaled\_scores}_{ij})}{\sum_{j'=1}^{T} \exp(\text{scaled\_scores}_{ij'})}$$

Each row of the weight matrix sums to 1. The weights $\text{weights}_{ij}$ tell us what fraction of its attention token $i$ should pay to token $j$.

**Step 4: Compute the output.**

$$\text{output}_i = \sum_{j=1}^{T} \text{weights}_{ij} \cdot v_j$$

The output for token $i$ is a weighted average of all value vectors, with weights determined by the query-key compatibility.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention.

    Q: (..., T, d_k)
    K: (..., T, d_k)
    V: (..., T, d_v)
    mask: (..., T, T) — 1 = attend, 0 = block
    """
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

In [ ]:
# Demonstrate on a tiny example: T=4 tokens, d_k=3
T, d_k = 4, 3
Q = torch.randn(T, d_k)
K = torch.randn(T, d_k)
V = torch.randn(T, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape:       {Q.shape}")
print(f"K shape:       {K.shape}")
print(f"V shape:       {V.shape}")
print(f"Output shape:  {output.shape}")
print(f"Weights shape: {weights.shape}")
print(f"\nAttention weights (each row sums to 1):")
print(weights.detach().numpy())
print(f"Row sums: {weights.sum(dim=-1).detach().numpy()}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights.detach().numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xlabel("Key position (attending to)")
ax.set_ylabel("Query position (attending from)")
ax.set_title("Attention weights")
ax.set_xticks(range(T))
ax.set_yticks(range(T))
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

Each row shows how one token distributes its attention across all positions. With random weights, the attention is roughly uniform — training is what makes attention patterns meaningful.

### Why Scale by $\sqrt{d_k}$?

Suppose the entries of $q_i$ and $k_j$ are drawn independently with mean 0 and variance 1. Their dot product is:

$$q_i \cdot k_j = \sum_{l=1}^{d_k} q_{il} \, k_{jl}$$

This is a sum of $d_k$ independent products, each with mean 0 and variance 1, so:

$$\mathbb{E}[q_i \cdot k_j] = 0, \qquad \text{Var}(q_i \cdot k_j) = d_k$$

For large $d_k$, the dot products become large in magnitude. This pushes the softmax into regions where its output is nearly one-hot — almost all the weight goes to a single token. In these saturated regions, the gradients of the softmax are extremely small, making learning slow or impossible.

Dividing by $\sqrt{d_k}$ rescales the variance back to 1:

$$\text{Var}\!\left(\frac{q_i \cdot k_j}{\sqrt{d_k}}\right) = \frac{d_k}{d_k} = 1$$

This keeps the softmax in a well-behaved regime where gradients flow effectively.

### Causal Masking

For **autoregressive language modeling** — predicting the next token given the previous tokens — we need to ensure that token $i$ can only attend to tokens at positions $j \leq i$. If token $i$ could attend to future tokens, it would "cheat" by looking at the answer it is supposed to predict.

We enforce this with a **causal mask**: a lower-triangular binary matrix applied before the softmax. Positions where the mask is 0 are set to $-\infty$, so they receive zero weight after the softmax:

$$\text{mask}_{ij} = \begin{cases} 1 & \text{if } j \leq i \\ 0 & \text{if } j > i \end{cases}$$

In [ ]:
T = 6
causal_mask = torch.tril(torch.ones(T, T))

Q = torch.randn(T, 8)
K = torch.randn(T, 8)
V = torch.randn(T, 8)

_, weights_no_mask = scaled_dot_product_attention(Q, K, V)
_, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im0 = axes[0].imshow(weights_no_mask.detach().numpy(), cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Without mask (bidirectional)")
axes[0].set_xlabel("Key position")
axes[0].set_ylabel("Query position")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(weights_causal.detach().numpy(), cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("With causal mask (autoregressive)")
axes[1].set_xlabel("Key position")
axes[1].set_ylabel("Query position")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

With the causal mask, the upper triangle is zero — each token can only attend to itself and the tokens that came before it. The first token can only attend to itself; the last token can attend to everything.

## Multi-Head Attention

A single attention head computes one set of attention weights — it can only capture one type of relationship at a time. In practice, different tokens need to attend to different things simultaneously: syntactic structure, semantic similarity, positional proximity, etc.

**Multi-head attention** runs $h$ attention heads in parallel, each with its own learned projections, then concatenates their outputs:

$$\text{head}_i = \text{Attention}(X W_Q^{(i)},\; X W_K^{(i)},\; X W_V^{(i)})$$

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W_O$$

where $W_Q^{(i)}, W_K^{(i)} \in \mathbb{R}^{d \times d_k}$, $W_V^{(i)} \in \mathbb{R}^{d \times d_v}$, and $W_O \in \mathbb{R}^{h d_v \times d}$.

To keep the total computation constant, we typically set $d_k = d_v = d / h$. Each head operates on a lower-dimensional projection, and the concatenated output has the same dimension as the input.

Different heads can specialize in different types of relationships. In practice, researchers have observed heads that track syntactic dependencies (subject-verb agreement), semantic roles, and local positional patterns.

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, d = x.shape
        # Project and reshape to (B, n_heads, T, d_k)
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product attention per head
        out, _ = scaled_dot_product_attention(Q, K, V, mask=mask)

        # Concatenate heads and project
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out)

In [ ]:
d_model, n_heads = 64, 4
mha = MultiHeadAttention(d_model, n_heads)

x = torch.randn(2, 10, d_model)  # batch=2, seq_len=10
out = mha(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"d_k per head: {mha.d_k}")
print(f"Parameters:   {sum(p.numel() for p in mha.parameters()):,}")

The output has the same shape as the input — multi-head attention is a sequence-to-sequence mapping that preserves dimensionality.

## Positional Encoding

Self-attention is **permutation-equivariant**: if we shuffle the input tokens, the output is shuffled in the same way. The attention mechanism has no inherent notion of position — it treats the sequence as a set.

To give the model information about token order, we add a **positional encoding** to each token embedding. Vaswani et al. (2017) use sinusoidal functions at different frequencies:

$$\text{PE}_{(t, 2i)} = \sin\!\left(\frac{t}{10000^{2i/d}}\right)$$

$$\text{PE}_{(t, 2i+1)} = \cos\!\left(\frac{t}{10000^{2i/d}}\right)$$

where $t$ is the position index and $i$ is the dimension index. The positional encoding is added to the token embeddings:

$$\tilde{x}_t = x_t + \text{PE}_t$$

This encoding has several desirable properties:

- Each position gets a unique encoding vector
- Nearby positions have similar encodings (high cosine similarity)
- The relative position between any two tokens can be expressed as a linear transformation of their encodings
- The encoding extends to sequence lengths not seen during training

In [ ]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


pe = positional_encoding(128, 64)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pe.numpy().T, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
ax.set_xlabel("Position")
ax.set_ylabel("Dimension")
ax.set_title("Sinusoidal Positional Encoding")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

The low-frequency dimensions (bottom) change slowly across positions, capturing coarse position information. The high-frequency dimensions (top) change rapidly, capturing fine-grained position differences.

In [ ]:
# Cosine similarity between position vectors
pe_norm = pe / pe.norm(dim=1, keepdim=True)
cos_sim = pe_norm[:32] @ pe_norm[:32].T

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cos_sim.numpy(), cmap="RdBu", vmin=-1, vmax=1)
ax.set_xlabel("Position")
ax.set_ylabel("Position")
ax.set_title("Cosine similarity between positional encodings")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

Nearby positions have high cosine similarity, and the similarity decreases smoothly with distance. This gives the model a natural sense of "closeness" between positions.

## The Transformer Block

A single transformer block combines four components:

1. **Multi-head self-attention** — lets each token gather information from all (visible) positions
2. **Feed-forward network (FFN)** — applies a nonlinear transformation independently to each position
3. **Layer normalization** — stabilizes training by normalizing activations
4. **Residual connections** — allow gradients to flow directly through the network

Using the **pre-norm** convention (LayerNorm before each sub-layer, which is more stable than the original post-norm):

$$\begin{align*}
X' &= X + \text{MultiHead}(\text{LayerNorm}(X)) \\
\text{out} &= X' + \text{FFN}(\text{LayerNorm}(X'))
\end{align*}$$

The feed-forward network is a two-layer MLP applied independently to each token:

$$\text{FFN}(x) = W_2 \, \text{GELU}(W_1 x + b_1) + b_2$$

where $W_1 \in \mathbb{R}^{d \times 4d}$ and $W_2 \in \mathbb{R}^{4d \times d}$. The inner dimension is conventionally $4 \times d$ — this gives the network more capacity to transform representations at each position.

The **residual connections** ($X + \ldots$) serve the same purpose as the cell state in LSTMs (L09): they create a "gradient highway" that allows gradients to flow backward through the network without being repeatedly multiplied by weight matrices. This is what makes it practical to stack many transformer blocks deep.

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.drop(self.attn(self.ln1(x), mask=mask))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x

In [ ]:
block = TransformerBlock(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
out = block(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Parameters:   {sum(p.numel() for p in block.parameters()):,}")

## Putting It Together: A GPT-Style Language Model

We now stack all the components into a complete **decoder-only transformer** — the architecture behind GPT, LLaMA, and Claude. The full model consists of:

1. **Token embedding**: `nn.Embedding(vocab_size, d_model)` — maps each token ID to a dense vector
2. **Positional encoding**: added to the token embeddings to inject position information
3. **$N$ transformer blocks**: the core computation, stacked sequentially
4. **Final layer norm**: stabilizes the output
5. **Linear output head**: projects from $d_{\text{model}}$ to `vocab_size` — produces logits over the vocabulary

The output at position $t$ is a probability distribution over the vocabulary for the **next** token:

$$P(x_{t+1} | x_1, \ldots, x_t) = \text{softmax}(W_{\text{out}} \, h_t^{(N)})$$

where $h_t^{(N)}$ is the representation of position $t$ after passing through all $N$ transformer blocks.

In [ ]:
class GPT(nn.Module):

    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_len,
                 d_ff=None, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer(
            "pe", positional_encoding(max_len, d_model).unsqueeze(0)
        )
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.max_len = max_len

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pe[:, :T, :]
        x = self.drop(x)

        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0)
        for block in self.blocks:
            x = block(x, mask=mask)

        x = self.ln_f(x)
        return self.head(x)  # (B, T, vocab_size)

In [ ]:
model = GPT(
    vocab_size=1000, d_model=128, n_heads=4,
    n_layers=4, max_len=256
)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

dummy = torch.randint(0, 1000, (2, 32))
logits = model(dummy)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {logits.shape}")

The model takes a batch of token ID sequences and outputs logits for the next token at every position. With ~1M parameters, this is a tiny model — GPT-2 has 124M, GPT-3 has 175B, and frontier models today have hundreds of billions.

## Pre-training Objectives

With the architecture in place, we need a training objective — a loss function that teaches the model the structure of language from raw text. Two dominant approaches exist:

### Next-token prediction (autoregressive LM)

Given a sequence $x_1, x_2, \ldots, x_T$, predict each token from its predecessors. The loss is the average cross-entropy:

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P_\theta(x_t \mid x_1, \ldots, x_{t-1})$$

The causal mask ensures that the prediction at position $t$ cannot see tokens at positions $> t$. This is the objective used by GPT, LLaMA, and Claude.

### Masked language modeling (MLM)

Randomly replace ~15% of tokens with a special `[MASK]` token, then predict the originals. The model sees **bidirectional** context (both left and right). The loss is computed only on the masked positions. This is the objective used by BERT.

We will use **next-token prediction** because it naturally supports text generation — each generated token is fed back as input for the next — and it is the foundation of all modern generative language models.

## Pre-training Data

Modern language models are pre-trained on massive corpora:

| Model | Training data | Tokens |
|---|---|---|
| GPT-2 (2019) | WebText (Reddit links) | ~10B |
| GPT-3 (2020) | Common Crawl + books + Wikipedia | ~300B |
| LLaMA 2 (2023) | Web + code + books | 2T |
| LLaMA 3 (2024) | Web + code + books | 15T |

Data quality matters as much as quantity. Key practices include:
- **Deduplication**: removing near-duplicate documents to prevent memorization
- **Filtering**: removing low-quality, toxic, or machine-generated text
- **Domain mixing**: balancing web text, books, code, and academic text

For our classroom exercise, we will train on a small economics corpus. The model will not be "intelligent" — it will merely learn statistical patterns in the text — but the process is identical to what happens at scale.

## Pre-training on Real Text

We will train our GPT model on text from Adam Smith's *The Wealth of Nations*. We use a character-level tokenizer for simplicity — the architecture is the same regardless of tokenizer choice.

In [ ]:
# --- Corpus ---
# Extended passages from The Wealth of Nations (public domain)
text = (
    "The annual labour of every nation is the fund which originally supplies "
    "it with all the necessaries and conveniences of life which it annually "
    "consumes, and which consist either in the immediate produce of that "
    "labour, or in what is purchased with that produce from other nations. "
    "According therefore as this produce, or what is purchased with it, bears "
    "a greater or smaller proportion to the number of those who are to consume "
    "it, the nation will be better or worse supplied with all the necessaries "
    "and conveniences for which it has occasion. But this proportion must in "
    "every nation be regulated by two different circumstances; first, by the "
    "skill, dexterity, and judgment with which its labour is generally applied; "
    "and, secondly, by the proportion between the number of those who are "
    "employed in useful labour, and that of those who are not so employed. "
    "Whatever be the soil, climate, or extent of territory of any particular "
    "nation, the abundance or scantiness of its annual supply must, in that "
    "particular situation, depend upon those two circumstances. "
    "The greatest improvement in the productive powers of labour, and the "
    "greater part of the skill, dexterity, and judgment with which it is "
    "anywhere directed, or applied, seem to have been the effects of the "
    "division of labour. The effects of the division of labour, in the general "
    "business of society, will be more easily understood by considering in "
    "what manner it operates in some particular manufactures. "
    "To take an example, therefore, from a very trifling manufacture; but one "
    "in which the division of labour has been very often taken notice of, the "
    "trade of the pin-maker; a workman not educated to this business, nor "
    "acquainted with the use of the machinery employed in it, could scarce, "
    "perhaps, with his utmost industry, make one pin in a day, and certainly "
    "could not make twenty. But in the way in which this business is now "
    "carried on, not only the whole work is a peculiar trade, but it is "
    "divided into a number of branches, of which the greater part are likewise "
    "a peculiar trade. One man draws out the wire, another straights it, a "
    "third cuts it, a fourth points it, a fifth grinds it at the top for "
    "receiving the head; to make the head requires two or three distinct "
    "operations; to put it on is a peculiar business, to whiten the pins is "
    "another; it is even a trade by itself to put them into the paper; and the "
    "important business of making a pin is, in this manner, divided into about "
    "eighteen distinct operations, which, in some manufactories, are all "
    "performed by distinct hands, though in others the same man will sometimes "
    "perform two or three of them. I have seen a small manufactory of this "
    "kind where ten men only were employed, and where some of them consequently "
    "performed two or three distinct operations. But though they were very poor, "
    "and therefore but indifferently accommodated with the necessary machinery, "
    "they could, when they exerted themselves, make among them about twelve "
    "pounds of pins in a day. There are in a pound upwards of four thousand "
    "pins of a middling size. Those ten persons, therefore, could make among "
    "them upwards of forty-eight thousand pins in a day. Each person, therefore, "
    "making a tenth part of forty-eight thousand pins, might be considered as "
    "making four thousand eight hundred pins in a day. But if they had all "
    "wrought separately and independently, and without any of them having been "
    "educated to this peculiar business, they certainly could not each of them "
    "have made twenty, perhaps not one pin in a day; that is, certainly, not "
    "the two hundred and fortieth, perhaps not the four thousand eight hundredth "
    "part of what they are at present capable of performing, in consequence of "
    "a proper division and combination of their different operations. "
    "In every other art and manufacture, the effects of the division of labour "
    "are similar to what they are in this very trifling one; though, in many "
    "of them, the labour can neither be so much subdivided, nor reduced to so "
    "great a simplicity of operation. The division of labour, however, so far "
    "as it can be introduced, occasions, in every art, a proportionable increase "
    "of the productive powers of labour. The separation of different trades and "
    "employments from one another seems to have taken place in consequence of "
    "this advantage. This separation, too, is generally carried furthest in "
    "those countries which enjoy the highest degree of industry and improvement; "
    "what is the work of one man in a rude state of society being generally "
    "that of several in an improved one. In every improved society, the farmer "
    "is generally nothing but a farmer; the manufacturer, nothing but a "
    "manufacturer. The labour, too, which is necessary to produce any one "
    "complete manufacture, is almost always divided among a great number of "
    "hands. How many different trades are employed in each branch of the linen "
    "and woollen manufactures, from the growers of the flax and the wool, to "
    "the bleachers and smoothers of the linen, or to the dyers and dressers "
    "of the cloth! The nature of agriculture, indeed, does not admit of so "
    "many subdivisions of labour, nor of so complete a separation of one "
    "business from another, as manufactures. It is impossible to separate so "
    "entirely the business of the grazier from that of the corn-farmer, as "
    "the trade of the carpenter is commonly separated from that of the smith. "
    "The spinner is almost always a distinct person from the weaver; but the "
    "ploughman, the harrower, the sower of the seed, and the reaper of the "
    "corn, are often the same. The occasions for those different sorts of "
    "labour returning only with the different seasons of the year, it is "
    "impossible for one man to be constantly employed in any one of them. This "
    "impossibility of making so complete and entire a separation of all the "
    "different branches of labour employed in agriculture is perhaps the "
    "reason why the improvement of the productive powers of labour in this "
    "art does not always keep pace with their improvement in manufactures. "
    "The most opulent nations, indeed, generally excel all their neighbours "
    "in agriculture as well as in manufactures; but they are commonly more "
    "distinguished by their superiority in the latter than in the former. "
    "This great increase of the quantity of work which, in consequence of the "
    "division of labour, the same number of people are capable of performing, "
    "is owing to three different circumstances; first, to the increase of "
    "dexterity in every particular workman; secondly, to the saving of the "
    "time which is commonly lost in passing from one species of work to "
    "another; and lastly, to the invention of a great number of machines "
    "which facilitate and abridge labour, and enable one man to do the work "
    "of many."
)

print(f"Corpus: {len(text):,} characters")

In [ ]:
# Character-level tokenizer
chars = sorted(set(text))
vocab_size = len(chars)
char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for ch, i in char_to_id.items()}

encode = lambda s: [char_to_id[c] for c in s]
decode = lambda ids: "".join(id_to_char[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
print(f"Vocab size: {vocab_size}")
print(f"Data shape: {data.shape}")
print(f"Vocabulary: {''.join(chars)}")

In [ ]:
# Train / val split (90/10 by position)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train: {len(train_data):,} tokens")
print(f"Val:   {len(val_data):,} tokens")

In [ ]:
class TextDataset(Dataset):
    """
    Sliding-window dataset for language modeling.
    Item i = (x, y) where
      x: (block_size,) — input token IDs
      y: (block_size,) — target token IDs (shifted by 1)
    """
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


BLOCK_SIZE = 64
BATCH_SIZE = 32

train_ds = TextDataset(train_data, BLOCK_SIZE)
val_ds = TextDataset(val_data, BLOCK_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train sequences: {len(train_ds):,}")
print(f"Val sequences:   {len(val_ds):,}")

x_ex, y_ex = train_ds[0]
print(f"x shape: {x_ex.shape}   y shape: {y_ex.shape}")
print(f"x[:20]:  {decode(x_ex[:20].tolist())}")
print(f"y[:20]:  {decode(y_ex[:20].tolist())}")

The target `y` is the input `x` shifted by one position — exactly the next-token prediction setup.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), y.view(-1)
            )
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)


def fit(model, n_epochs=30, lr=3e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_hist, val_hist = [], []
    for epoch in range(1, n_epochs + 1):
        tl = run_epoch(model, train_loader, optimizer)
        vl = run_epoch(model, val_loader)
        train_hist.append(tl)
        val_hist.append(vl)
        if epoch % 10 == 0 or epoch == 1:
            print(f"  epoch {epoch:3d}/{n_epochs}  train={tl:.4f}  val={vl:.4f}")
    return train_hist, val_hist

In [ ]:
model = GPT(
    vocab_size=vocab_size,
    d_model=64,
    n_heads=4,
    n_layers=4,
    max_len=BLOCK_SIZE,
    dropout=0.1,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Training...\n")

train_hist, val_hist = fit(model, n_epochs=50, lr=3e-3)

In [ ]:
epochs = range(1, len(train_hist) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, train_hist, label="Train", color="steelblue")
ax.plot(epochs, val_hist, label="Val", color="darkorange")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Pre-training loss")
ax.legend()
plt.tight_layout()
plt.show()

## Text Generation

Once pre-trained, we can generate text **autoregressively**: sample the next token from the model's predicted distribution, append it to the sequence, and repeat.

A **temperature** parameter $\tau > 0$ controls the diversity of the generated text. Given the logits $z$ from the model, the sampling distribution is:

$$P(x_{t+1} = v) = \frac{\exp(z_v / \tau)}{\sum_{v'} \exp(z_{v'} / \tau)}$$

- $\tau \to 0$: greedy decoding — always pick the most likely token (deterministic, repetitive)
- $\tau = 1$: sample from the model's learned distribution
- $\tau > 1$: flatter distribution — more random, more diverse

In [ ]:
@torch.no_grad()
def generate(model, context, max_new_tokens, temperature=1.0):
    model.eval()
    idx = context.clone()
    for _ in range(max_new_tokens):
        # Crop to max_len if needed
        idx_cond = idx[:, -model.max_len:]
        logits = model(idx_cond)
        # Logits for the last position
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

In [ ]:
model.to(device)

prompt = "The division of labour"
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

for temp in [0.5, 1.0, 1.5]:
    generated = generate(model, context, max_new_tokens=150, temperature=temp)
    text_out = decode(generated[0].tolist())
    print(f"--- Temperature = {temp} ---")
    print(text_out)
    print()

The generated text will not be coherent — our model is tiny and our corpus is small. But notice that the model has learned basic patterns: common words, character-level structure, and rough stylistic features of the source text. At scale, the same architecture trained on trillions of tokens produces remarkably fluent text.

## Scaling Laws

One of the most striking empirical findings in deep learning is that language model performance follows **power-law scaling** with respect to three factors:

1. **Parameters** ($N$): the number of trainable weights
2. **Data** ($D$): the number of training tokens
3. **Compute** ($C$): the total floating-point operations used for training

Kaplan et al. (2020) found that the cross-entropy loss $L$ decreases as a power law:

$$L(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N}, \qquad L(D) \approx \left(\frac{D_c}{D}\right)^{\alpha_D}$$

where $\alpha_N \approx 0.076$ and $\alpha_D \approx 0.095$ are empirically determined exponents.

Hoffmann et al. (2022) refined these findings with the **Chinchilla scaling law**: for a fixed compute budget, the optimal strategy allocates compute equally between model size and data. Specifically, if compute doubles, both the model size and the number of training tokens should double. This means a 10x larger model should be trained on 10x more data — not the same data for longer.

These scaling laws are relevant to economics: they describe an **optimization problem under resource constraints** (compute is scarce and expensive) with diminishing but predictable returns. The decision of how large a model to train and how much data to collect is fundamentally an economic one.

## Transformer vs. RNN

| Property | RNN / LSTM | Transformer |
|---|---|---|
| **Training parallelism** | Sequential (each step depends on the last) | Fully parallel across positions |
| **Long-range dependencies** | Through chain of hidden states (information decays) | Direct attention in constant depth |
| **Memory (training)** | $O(T)$ — one hidden state per step | $O(T^2)$ — full attention matrix |
| **Memory (inference)** | $O(1)$ per step (fixed hidden state) | $O(T)$ per step (KV cache) |
| **Training speed** | Slow (inherently sequential) | Fast (parallelizable on GPUs) |
| **Dominant use** | Pre-2017 sequence modeling | Nearly all modern language models |

The $O(T^2)$ memory cost of attention is the main limitation of transformers. For a sequence of $T$ tokens, the attention matrix has $T^2$ entries. This is why language models have **context windows** — a maximum sequence length beyond which they cannot operate. Current frontier models have context windows of 128k–1M tokens, achieved through engineering optimizations on the basic attention mechanism.

## References

- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017). Attention is all you need. *NeurIPS 2017*.
- Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. *OpenAI Technical Report*.
- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL 2019*.
- Kaplan, J., McCandlish, S., Henighan, T., Brown, T. B., Chess, B., Child, R., Gray, S., Radford, A., Wu, J., & Amodei, D. (2020). Scaling laws for neural language models. *arXiv:2001.08361*.
- Hoffmann, J., Borgeaud, S., Mensch, A., et al. (2022). Training compute-optimal large language models. *NeurIPS 2022*.